In [ ]:
import re
import pandas as pd

# -----------------------
# Local file paths
# -----------------------
B1A_STRUCT_PATH = "dict/b1a_series.csv"
CE_SERIES_PATH = "dict/ce.series"
CE_INDUSTRY_PATH = "dict/ce.industry"

def strip_df(df):
    df.columns = df.columns.str.strip()
    return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

def norm(s):
    s = "" if s is None else str(s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s&/-]", "", s)
    return s

def clean_naics(s):
    return "" if s is None else re.sub(r"\s+", "", str(s).strip())

# -----------------------
# Load your B-1a structure
# -----------------------
b1a = strip_df(pd.read_csv(B1A_STRUCT_PATH, dtype=str))
b1a["row_order"] = pd.to_numeric(b1a["row_order"], errors="coerce")
b1a["industry_name"] = b1a["industry_name"].astype(str).str.strip()
b1a["industry_name_norm"] = b1a["industry_name"].apply(norm)
b1a["naics_code"] = b1a["naics_code"].fillna("").apply(clean_naics)
b1a = b1a.sort_values("row_order")

# -----------------------
# Load CES catalogs
# -----------------------
ce_ser = strip_df(pd.read_csv(CE_SERIES_PATH, sep="\t", dtype=str))
ce_ind = strip_df(pd.read_csv(CE_INDUSTRY_PATH, sep="\t", dtype=str))

ce_ind["naics_code"] = ce_ind["naics_code"].fillna("").apply(clean_naics)
ce_ind["industry_name_norm"] = ce_ind["industry_name"].apply(norm)

# -----------------------
# Restrict to B-1a concept
#   CES + SA + All employees (01)
# -----------------------
ce_ser = ce_ser[
    ce_ser["series_id"].str.startswith("CES", na=False)
    & ce_ser["seasonal"].eq("S")
    & ce_ser["data_type_code"].eq("01")
]

ce_ser = ce_ser.merge(
    ce_ind[["industry_code", "naics_code", "industry_name", "industry_name_norm", "display_level"]],
    on="industry_code",
    how="left"
)

# -----------------------
# 1) Match by NAICS (preferred)
# -----------------------
m_naics = b1a[b1a["naics_code"] != ""].merge(
    ce_ser,
    on="naics_code",
    how="left"
)

# -----------------------
# 2) Match aggregates by industry_name
# -----------------------
blank = b1a[b1a["naics_code"] == ""].copy()

m_name = blank.merge(
    ce_ser,
    on="industry_name_norm",
    how="left"
)

# -----------------------
# Combine and finalize
# -----------------------
mapping = pd.concat([m_naics, m_name], ignore_index=True)
mapping = mapping.sort_values("row_order").reset_index(drop=True)
mapping["matched"] = mapping["series_id"].notna()


print("Rows:", len(mapping))
print("Matched:", int(mapping["matched"].sum()))
print("Unmatched:", int((~mapping["matched"]).sum()))

Rows: 844
Matched: 844
Unmatched: 0


/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_64480/3356051652.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_64480/3356051652.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/var/folders/x3/1282rh0s7_b240x5mjx1zlmc0000gn/T/ipykernel_64480/3356051652.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [54]:
mismatch = mapping[
    mapping["industry_name_norm_x"].notna() &
    mapping["industry_name_norm_y"].notna() &
    (mapping["industry_name_norm_x"] != mapping["industry_name_norm_y"])
].copy()

print("Mismatches:", len(mismatch))
print(
    mismatch[[
        "row_order",
        "industry_name_x",
        "industry_name_y",
        "industry_name_norm_x",
        "industry_name_norm_y",
        "series_id"
    ]].to_string(index=False)
)

Mismatches: 2
 row_order                            industry_name_x                            industry_name_y                       industry_name_norm_x                       industry_name_norm_y     series_id
        45    Residential specialty trade contractors Nonresidential specialty trade contractors    residential specialty trade contractors nonresidential specialty trade contractors CES2023800201
        46 Nonresidential specialty trade contractors    Residential specialty trade contractors nonresidential specialty trade contractors    residential specialty trade contractors CES2023800101


In [55]:
mismatch = (
    mapping["industry_name_norm_x"].notna()
    & mapping["industry_name_norm_y"].notna()
    & (mapping["industry_name_norm_x"] != mapping["industry_name_norm_y"])
)
mapping = mapping.loc[~mismatch].copy()

# --- Step 2: fill missing industry_name_norm_x from industry_name_norm (NOT _y) ---
# (This assumes you currently have a column named exactly 'industry_name_norm'.)
mapping["industry_name_norm_x"] = mapping["industry_name_norm_x"].combine_first(mapping["industry_name_norm"])

# drop the helper column industry_name_norm
mapping = mapping.drop(columns=["industry_name_norm"], errors="ignore")

# drop all *_y columns
y_cols = [c for c in mapping.columns if c.endswith("_y")]
mapping = mapping.drop(columns=y_cols, errors="ignore")

# rename *_x columns by removing the suffix
mapping = mapping.rename(columns={c: c[:-2] for c in mapping.columns if c.endswith("_x")})

In [58]:
mapping.to_csv("b1a_mapping.csv", index=False)
mapping

,row_order,industry_name,naics_code,industry_name_norm,series_id,supersector_code,industry_code,data_type_code,seasonal,series_title,footnote_codes,begin_year,begin_period,end_year,end_period,display_level,naics_code,matched
0,1,Total nonfarm,NaN,total nonfarm,CES0000000001,00,00000000,01,S,"All employees, thousands, total nonfarm, seaso...",NaN,1939,M01,2026,M01,0,,True
1,2,Total private,NaN,total private,CES0500000001,05,05000000,01,S,"All employees, thousands, total private, seaso...",NaN,1939,M01,2026,M01,1,,True
2,3,Goods-producing,NaN,goods-producing,CES0600000001,06,06000000,01,S,"All employees, thousands, goods-producing, sea...",NaN,1939,M01,2026,M01,1,,True
3,4,Mining and logging,NaN,mining and logging,CES1000000001,10,10000000,01,S,"All employees, thousands, mining and logging, ...",NaN,1939,M01,2026,M01,2,,True
4,5,Logging,1133,logging,CES1011330001,10,10113300,01,S,"All employees, thousands, logging, seasonally ...",NaN,1947,M01,2026,M01,5,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
839,838,Local government utilities,NaN,local government utilities,CES9093222101,90,90932221,01,S,"All employees, thousands, local government uti...",I,1990,M01,2025,M12,5,,True
840,839,Local government transportation,NaN,local government transportation,CES9093248001,90,90932480,01,S,"All employees, thousands, local government tra...",I,1990,M01,2025,M12,5,,True
841,840,Local hospitals,NaN,local hospitals,CES9093262201,90,90932622,01,S,"All employees, thousands, local hospitals, sea...",I,1972,M01,2025,M12,5,,True
842,841,Local government general administration,NaN,local government general administration,CES9093292001,90,90932920,01,S,"All employees, thousands, local government gen...",I,1990,M01,2025,M12,5,,True
